
# LlamaIndex Chunking Experiments

This notebook mirrors the chunking comparison script so you can tweak parameters and re-run experiments interactively.



## Setup
Setting up the required paths.


In [9]:
import os
try:
    if "google.colab" in str(get_ipython()):
        print("Running on Colab, trying to find optimal path for project_root")

        from google.colab import drive
        drive.mount('/content/drive')

        # try Shravankumar's drive
        expected_path = "/content/drive/MyDrive/Colab Notebooks/SJSU/Semester 2/DATA-236/Homework-3"

        if os.path.exists(expected_path):
            print(f"Picking shravan's project dir {expected_path}")
            project_root = expected_path
        else:
            print("Using Colab local path")
            project_root = "/content"
    else:
        raise NameError("Not running in Colab")
except Exception:
    print("Running outside Colab, using local default.")
    project_root = os.path.abspath("./")

os.makedirs(project_root, exist_ok=True)

print("Project root:", project_root)

Running outside Colab, using local default.
Project root: /home/cloud_user/DATA-236-Projects/hw3/Chunking


In [ ]:

!pip3 install -U pip
try:
    import torch
    cuda_available = torch.cuda.is_available()
except ImportError:
    cuda_available = False
print("CUDA available:", cuda_available)
if cuda_available:
    print("CUDA is available, installing GPU-enabled PyTorch...")
    !pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
else:
    print("CUDA not detected, installing CPU-only PyTorch...")
    !pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip3 install sentence-transformers "llama-index" "llama-index-embeddings-huggingface" faiss-cpu numpy pandas tqdm scikit-learn requests
import torch
print("Torch version:", torch.__version__)
print("CUDA available in torch:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 14.5 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.0.1
    Uninstalling pip-25.0.1:
      Successfully uninstalled pip-25.0.1
CUDA available: False
CUDA not detected, installing CPU-only PyTorch...
Looking in indexes: https://download.pytorch.org/whl/cpu
INFO: pip is looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 64.0 MB/s  0:00:00
   ━━━━━━━━━━

## Install requirements

Install requirements into the kernel before running the cells.

In [12]:
import time
from pathlib import Path
import numpy as np
from typing import Iterable, Optional, Tuple, Dict, Any
import requests
from tqdm.auto import tqdm

import numpy as np

from llama_index.core import Document, VectorStoreIndex, StorageContext, Settings, load_index_from_storage
from llama_index.core.node_parser import (
    TokenTextSplitter,
    SemanticSplitterNodeParser,
    SentenceWindowNodeParser,
)
from llama_index.core.vector_stores import SimpleVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding


## Metadata Setup
Constants to download the required tiny_shakespeare file.

In [13]:

DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
DATA_FILE = Path("tiny_shakespeare.txt")
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"


## Helper functions
There are helper functions to dowload the file, setup the VectorStoreIndex and fetch the matching chunks based on the query.
The `retrieve_and_print_results` prints a comprehensive summary of the results.

In [14]:

## Helper functions to run the code.
def download_file(url: str, dest: Path) -> None:
    if dest.exists():
        print(f"Using existing {dest}")
        return
    print(f"Downloading {url} -> {dest}")
    resp = requests.get(url, stream=True)
    resp.raise_for_status()
    total = int(resp.headers.get("content-length", 0))
    with dest.open("wb") as f:
        for chunk in tqdm(resp.iter_content(chunk_size=8192), total=total // 8192 or None):
            if chunk:
                f.write(chunk)


def _index_exists(persist_dir: str) -> bool:
    p = Path(persist_dir)
    required = ["docstore.json", "index_store.json"]
    return p.exists() and all((p / name).exists() for name in required)

def create_or_load_index(
    text: str,
    splitter,
    embed_model,
    technique_name: str,
    persist_dir: str,
    force: bool = False
) -> Tuple[VectorStoreIndex, Optional[list]]:
    """
    Build and persist an index, or load it if it already exists.
    Returns (index, nodes). nodes is None when loading.
    """
    persist_path = Path(persist_dir)
    if force and persist_path.exists():
        for f in persist_path.glob("*"):
            try: f.unlink()
            except IsADirectoryError:
                for g in f.glob("*"): g.unlink()
                f.rmdir()

    if not force and _index_exists(persist_dir):
        print(f"{technique_name}: found persisted index at {persist_dir}, loading...")
        storage_context = StorageContext.from_defaults(persist_dir=persist_dir)
        index = load_index_from_storage(storage_context, embed_model=embed_model)
        nodes = None
        print(f"{technique_name}: loaded.")
        return index, nodes

    print(f"{technique_name}: building and persisting to {persist_dir} ...")
    document = Document(text=text)
    nodes = splitter.get_nodes_from_documents([document])
    print(f"   Created {len(nodes)} nodes")

    storage_context = StorageContext.from_defaults()
    index = VectorStoreIndex(
        nodes,
        vector_store=SimpleVectorStore(),
        embed_model=embed_model,
        storage_context=storage_context,
    )
    persist_path.mkdir(parents=True, exist_ok=True)
    storage_context.persist(persist_dir=persist_dir)
    print(f"{technique_name}: persisted at {persist_dir}\n")
    return index, nodes


def combined_text(node) -> str:
    """Combine node.text with useful metadata (window/original_text/context_str)."""
    base = (getattr(node, "text", "") or "").strip()
    md = getattr(node, "metadata", {}) or {}
    parts = []
    for k in ("window", "original_text", "context_str"):
        v = md.get(k)
        if isinstance(v, str) and v.strip():
            parts.append(v.strip())
    return f"{base} [window] {' '.join(parts)}" if parts else base

def truncate(s: str, n: int = 160) -> str:
    s = (s or "").replace("\n", " ").strip()
    return s if len(s) <= n else s[: n - 1] + "…"

def cosine(a: np.ndarray, b: np.ndarray) -> float:
    an = np.linalg.norm(a); bn = np.linalg.norm(b)
    if an == 0.0 or bn == 0.0: return 0.0
    return float(np.dot(a, b) / (an * bn))

def _maybe_batch_embed(embed_model, texts: list[str]) -> list[np.ndarray]:
    """Use batch embedding if available; otherwise fall back to per-text."""
    if hasattr(embed_model, "get_text_embedding_batch"):
        vecs = embed_model.get_text_embedding_batch(texts)
        return [np.array(v, dtype=np.float32) for v in vecs]
    else:
        return [np.array(embed_model.get_text_embedding(t), dtype=np.float32) for t in texts]

def retrieve_and_print_results(
    index,
    query: str,
    technique_name: str,
    embed_model,
    top_k: int = 5,
    ensure_terms: Optional[Iterable[str]] = None,
    ensure_pool_size: int = 48,
    all_nodes: Optional[Iterable[Any]] = None,
    retriever=None,
    embedding_cache: Optional[Dict[str, np.ndarray]] = None,
) -> Tuple[list, float]:
    """
    Pretty, fast retrieval with optional exact-term preference.

    - If `retriever` is None, uses index.as_retriever(similarity_top_k=pool).
    - `embedding_cache` lets you reuse node embeddings across calls: pass a dict {} once and keep it around.
    - If `ensure_terms` is provided, we enlarge the pool and prefer rows containing all terms (case-insensitive).

    Returns: (selected_rows, latency_ms)
    Each row contains: original_rank, store_score, cosine_sim, chunk_len, preview, has_terms, node
    """
    ensure_terms = [t.lower() for t in (ensure_terms or [])]
    pool = max(top_k, ensure_pool_size) if ensure_terms else top_k
    cache = embedding_cache if embedding_cache is not None else {}

    q_vec = np.array(embed_model.get_text_embedding(query), dtype=np.float32)

    retriever = retriever or index.as_retriever(similarity_top_k=pool)

    t0 = time.time()
    nodes = retriever.retrieve(query)
    latency_ms = (time.time() - t0) * 1000.0

    print(f"=== {technique_name} Retrieval Results ===")
    print(f"Query: {query!r}")
    print(f"Retrieved {len(nodes)} nodes in {latency_ms:.2f} ms (pool={pool})")

    def key_for(node) -> str:
        nid = getattr(node, "node_id", None) or getattr(node, "id_", None)
        return str(nid) if nid is not None else f"text::{hash(getattr(node, 'text', '') or '')}"

    missing = []
    order_keys = []
    texts_for_missing = []
    for node in nodes:
        k = key_for(node)
        order_keys.append(k)
        if k not in cache:
            missing.append(node)
            texts_for_missing.append(getattr(node, "text", "") or "")

    if missing:
        new_vecs = _maybe_batch_embed(embed_model, texts_for_missing)
        for k, v in zip((key_for(n) for n in missing), new_vecs):
            cache[k] = v

    rows = []
    for i, node in enumerate(nodes, start=1):
        combo = combined_text(node)
        has_terms = all(t in combo.lower() for t in ensure_terms) if ensure_terms else False
        d_vec = cache[key_for(node)]
        rows.append({
            "original_rank": i,
            "store_score": float(getattr(node, "score", 0.0) or 0.0),
            "cosine_sim": cosine(q_vec, d_vec),
            "chunk_len": len(getattr(node, "text", "") or ""),
            "preview": truncate(combo, 160),
            "has_terms": has_terms,
            "node": node,
        })

    # Rank
    if ensure_terms:
        rows.sort(key=lambda r: (int(r["has_terms"]), r["cosine_sim"]), reverse=True)
    else:
        rows.sort(key=lambda r: r["cosine_sim"], reverse=True)

    selected = rows[:top_k]

    # Safety net: make sure at least one terms-hit is included if available
    if ensure_terms and selected and not any(r["has_terms"] for r in selected):
        fb = next((r for r in rows if r["has_terms"]), None)
        if fb:
            if len(selected) >= top_k: selected[-1] = fb
            else: selected.append(fb)
        elif all_nodes:
            for n in all_nodes:
                combo = combined_text(n).lower()
                if all(t in combo for t in ensure_terms):
                    k = key_for(n)
                    if k not in cache:
                        cache[k] = np.array(embed_model.get_text_embedding(getattr(n, "text", "") or ""), dtype=np.float32)
                    selected.append({
                        "original_rank": 0,
                        "store_score": 0.0,
                        "cosine_sim": cosine(q_vec, cache[k]),
                        "chunk_len": len(getattr(n, "text", "") or ""),
                        "preview": truncate(combined_text(n), 160),
                        "has_terms": True,
                        "node": n,
                    })
                    break
        selected.sort(key=lambda r: (int(r["has_terms"]), r["cosine_sim"]), reverse=True)

    print(f"{'Rank':<4} {'Orig':<5} {'Store':<10} {'Cosine':<8} {'Terms':<5} Preview")
    for rank, r in enumerate(selected, start=1):
        print(f"{rank:<4} {r['original_rank']:<5} {r['store_score']:<10.4f} {r['cosine_sim']:<8.4f} "
              f"{('Y' if r['has_terms'] else 'N'):<5} {r['preview']}")

    if ensure_terms and not any(r["has_terms"] for r in selected):
        print("Warning Note: required terms were not present in the top-k results.")

    return selected, latency_ms


## Download the file in to local dir

In [15]:
download_file(DATA_URL, DATA_FILE)
text = DATA_FILE.read_text(encoding="utf-8")
print(f"Full dataset length: {len(text):,} characters")


Using existing tiny_shakespeare.txt
Full dataset length: 1,115,394 characters


## Load the `sentence-transformers/all-MiniLM-L6-v2` Model
This is done to interface llamaindex with HuggingFaceEmbedding API, it acts as a bridge between the sentence-transformers used for chunking from Huggingface and llamaIndex.


In [17]:
print(f"Loading embedding model: {MODEL_NAME}")
if cuda_available:
    embed_model = HuggingFaceEmbedding(model_name=MODEL_NAME, device="cuda", embed_batch_size=128)
else:
    embed_model = HuggingFaceEmbedding(model_name=MODEL_NAME, device="cpu", embed_batch_size=128)
Settings.embed_model = embed_model

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


## Create the splitters and index them in the vector store.

In [18]:

# Create splitters and indexes

token_splitter = TokenTextSplitter(chunk_size=512, chunk_overlap=64)
semantic_splitter = SemanticSplitterNodeParser(
    embed_model=embed_model,
    buffer_size=2,
    breakpoint_percentile_threshold=90,
)
sentence_splitter = SentenceWindowNodeParser(
    window_size=3,
    window_metadata_key="window",
    original_text_metadata_key="original_text",
)
token_dir = f"{project_root}/index_token"
semantic_dir = f"{project_root}/index_semantic"
sentence_dir = f"{project_root}/index_sentencewin"

token_index, token_nodes = create_or_load_index(text, token_splitter, embed_model, "Token-based", token_dir)
semantic_index, semantic_nodes = create_or_load_index(text, semantic_splitter, embed_model, "Semantic", semantic_dir)
sentence_index, sentence_nodes = create_or_load_index(text, sentence_splitter, embed_model, "Sentence Window", sentence_dir)


Token-based: building and persisting to /home/cloud_user/DATA-236-Projects/hw3/Chunking/index_token ...
   Created 677 nodes
Token-based: persisted at /home/cloud_user/DATA-236-Projects/hw3/Chunking/index_token

Semantic: building and persisting to /home/cloud_user/DATA-236-Projects/hw3/Chunking/index_semantic ...
   Created 1247 nodes
Semantic: persisted at /home/cloud_user/DATA-236-Projects/hw3/Chunking/index_semantic

Sentence Window: building and persisting to /home/cloud_user/DATA-236-Projects/hw3/Chunking/index_sentencewin ...
   Created 12453 nodes
Sentence Window: persisted at /home/cloud_user/DATA-236-Projects/hw3/Chunking/index_sentencewin



## Semantic Search (RAG's Retreval step)
Try to do a similarity search based on the different indexes we created and see if the expected terms are in the resulting matches, also compute scores and rank them.

In [19]:

query = "Who are the two feuding houses?"
required_terms = ["montague", "capulet"]

print("=== Main Query ===")
print(query)

common_kwargs = dict(
    query=query,
    embed_model=embed_model,
    top_k=5,
    ensure_terms=required_terms,
    ensure_pool_size=25,
)

print("\n" + "="*80)
retrieve_and_print_results(
    token_index,
    technique_name="Token-based",
    all_nodes=token_nodes,
    **common_kwargs
)

print("\n" + "="*80)
retrieve_and_print_results(
    semantic_index,
    technique_name="Semantic",
    all_nodes=semantic_nodes,
    **common_kwargs
)

print("\n" + "="*80)
retrieve_and_print_results(
    sentence_index,
    technique_name="Sentence Window",
    all_nodes=sentence_nodes,
    **common_kwargs
)


=== Main Query ===
Who are the two feuding houses?

=== Token-based Retrieval Results ===
Query: 'Who are the two feuding houses?'
Retrieved 25 nodes in 54.26 ms (pool=25)
Rank Orig  Store      Cosine   Terms Preview
1    6     0.2796     0.2796   Y     enemies to peace, Profaners of this neighbour-stained steel,-- Will they not hear? What, ho! you men, you beasts, That quench the fire of your pernicious rage …
2    1     0.3497     0.3497   N     side?  DERBY: John Duke of Norfolk, Walter Lord Ferrers, Sir Robert Brakenbury, and Sir William Brandon.  RICHMOND: Inter their bodies as becomes their births:…
3    2     0.3354     0.3354   N     would you say ye were beaten out of door; And rail upon the hostess of the house; And say you would present her at the leet, Because she brought stone jugs and…
4    3     0.3094     0.3094   N     is warm and new cut off, Write in the dust this sentence with thy blood, 'Wind-changing Warwick now can change no more.'  WARWICK: O cheerful colours! s

([{'original_rank': 4,
   'store_score': 0.456476278908496,
   'cosine_sim': 0.4564763307571411,
   'chunk_len': 21,
   'preview': "ROMEO: Whose house? [window] Servant: Up.   ROMEO: Whither?   Servant: To supper; to our house.   ROMEO: Whose house?   Servant: My master's.   ROMEO: Indeed, …",
   'has_terms': True,
   'node': NodeWithScore(node=TextNode(id_='13518f31-2554-4872-9a2f-f5256020fb27', embedding=None, metadata={'window': "Servant:\nUp.\n\n ROMEO:\nWhither?\n\n Servant:\nTo supper; to our house.\n\n ROMEO:\nWhose house?\n\n Servant:\nMy master's.\n\n ROMEO:\nIndeed, I should have ask'd you that before.\n\n Servant:\nNow I'll tell you without asking: my master is the\ngreat rich Capulet; and if you be not of the house\nof Montagues, I pray, come and crush a cup of wine.\n", 'original_text': 'ROMEO:\nWhose house?\n\n'}, excluded_embed_metadata_keys=['window', 'original_text'], excluded_llm_metadata_keys=['window', 'original_text'], relationships={<NodeRelationship.SOURCE: '1'>:

## Additional Quries and their results.

In [20]:
additional_queries = [
    ("Who is Romeo in love with?", ["romeo", "juliet"]),
    ("Which play contains the line 'To be, or not to be'?", ["to be, or not to be", "to be", "not to be"]),
]
common_kwargs = dict(
    embed_model=embed_model,
    top_k=5,
    ensure_pool_size=25,
)
for extra_query, terms in additional_queries:
    print("\n" + "="*80)
    print("=== Additional Query ===")
    print(extra_query)
    print("-"*80)

    retrieve_and_print_results(
        token_index, extra_query, "Token-based",
        ensure_terms=terms, all_nodes=token_nodes,
        **common_kwargs
    )
    print("-"*80)

    retrieve_and_print_results(
        semantic_index, extra_query, "Semantic",
        ensure_terms=terms, all_nodes=semantic_nodes,
        **common_kwargs
    )
    print("-"*80)

    retrieve_and_print_results(
        sentence_index, extra_query, "Sentence Window",
        ensure_terms=terms, all_nodes=sentence_nodes,
        **common_kwargs
    )


=== Additional Query ===
Who is Romeo in love with?
--------------------------------------------------------------------------------
=== Token-based Retrieval Results ===
Query: 'Who is Romeo in love with?'
Retrieved 25 nodes in 57.32 ms (pool=25)
Rank Orig  Store      Cosine   Terms Preview
1    1     0.6003     0.6003   Y     a Montague; The only son of your great enemy.  JULIET: My only love sprung from my only hate! Too early seen unknown, and known too late! Prodigious birth of l…
2    4     0.5338     0.5338   Y     I hope, thou wilt be satisfied.  JULIET: Indeed, I never shall be satisfied With Romeo, till I behold him--dead-- Is my poor heart for a kinsman vex'd. Madam, …
3    5     0.5261     0.5261   Y     a kinsman to the Montague; Affection makes him false; he speaks not true: Some twenty of them fought in this black strife, And all those twenty could but kill …
4    6     0.5144     0.5144   Y     love-devouring death do what he dare; It is enough I may but call her mine.